In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from qutip import *
import numpy as np
from scipy.special import jn_zeros

def h_t(t, args):
    h = args['h']
    h0 = args['h0']
    w = args['omega']
    return h0 + h * np.cos(w * t)

def h0_ham_lmg(N, n_ph, g, hbar, omega0, Jvalue):

    S = N/2

    # collective spin operators
    Sx = jmat(S, 'x')
    Sz = jmat(S, 'z')
    Sz2 = Sz * Sz

    dim_spin = int(2*S + 1)
    I_spin = qeye(dim_spin)

    # cavity
    a = destroy(n_ph)
    adag = a.dag()
    I_ph = qeye(n_ph)

    # -------- tensor lift --------
    Sx_full = tensor(Sx, I_ph)
    Sz_full = tensor(Sz, I_ph)
    Sz2_full = tensor(Sz2, I_ph)

    a_full = tensor(I_spin, a)
    adag_full = tensor(I_spin, adag)

    # -------- Hamiltonian parts --------

    # LMG interaction
    H0 = -2/(N-1) * Sz2_full

    # transverse field (time-dependent part will multiply this)
    H1 = 2 * Sx_full

    # cavity energy
    H2 = hbar * omega0 /(N-1) * (adag_full * a_full)

    # light-matter coupling
    H_int = (2 * g / np.sqrt(N)) * (a_full + adag_full) * Sx_full

    return H0, H1, H2, H_int, Sx_full, Sz_full, Sx


def run_dynamics_lmg(args):
    tlist = args['tlist']
    N = args['N']
    n_ph = args['n_ph']
    g = args['g']
    hbar = args['hbar']
    omega0 = args['omega0']
    Jvalue = args['Jvalue']

    
    H0, H1, H2, H_int, Sx_full, Sz_full, Sx = h0_ham_lmg(N, n_ph, g, hbar, omega0, Jvalue)
    H_td = [H0 + H2 + H_int, [H1,h_t]]
    en, sts = Sx.eigenstates() 
    rho = sts[0]
    psi_ph = basis(n_ph,0)
    rho0 = tensor(rho, psi_ph)
    results = mesolve(H_td, rho0, tlist, [], [], args = args, options=args['opts'])
    psi_ts = results.states

    E0 = np.array(expect(H0, psi_ts))
    E1 = np.array(expect(H1, psi_ts))
    drive_vals = args['h0'] + args['h'] * np.cos(args['omega'] * tlist)
    energies_spin = (E0 + drive_vals * E1)/N
    H_spin = np.sqrt(np.mean(energies_spin**2) - np.mean(energies_spin)**2)

    E2 = np.array(expect(H2, psi_ts))
    energies_ph = E2
    H_ph = np.sqrt(np.mean(energies_ph**2) - np.mean(energies_ph)**2)

    avg_ph_energy = np.mean(energies_ph)

    return H_spin, H_ph, avg_ph_energy